# 05 — Agent 3 : verbalisation des portions

L'Agent 3 transforme une représentation structurée en texte naturel.

Il ne doit :
- ni recalculer les métriques ;
- ni ajouter de nouvelles caractéristiques ;
- ni inventer de lieux ou de paysages.

Son rôle est uniquement de verbaliser.

In [ ]:
import json
import requests

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.1:8b"

def ask_ollama(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": {"temperature": 0},
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["message"]["content"].strip()

## 1. Verbalisation d'une portion

In [ ]:
def humanize_portion(portion):
        payload = {
            "start_m": portion["start_m"],
            "end_m": portion["end_m"],
            "annotations": portion["annotations"],
            "metrics": portion.get("metrics", {}),
        }

        prompt = f'''
Tu es chargé de décrire un itinéraire pédestre.

Transforme les données structurées ci-dessous en UNE phrase française naturelle,
factuelle et concise.

Règles :
- n'ajoute aucune information absente ;
- n'invente aucun lieu ;
- n'invente aucune végétation ;
- n'invente aucun panorama ;
- ne modifie pas les distances ;
- si plusieurs annotations sont présentes, combine-les dans une phrase fluide ;
- évite le jargon technique inutile.

Données :
{json.dumps(payload, ensure_ascii=False)}

Réponds uniquement avec la phrase finale.
'''

        return ask_ollama(prompt)

## 2. Exemple

In [ ]:
portion = {
    "start_m": 1250,
    "end_m": 1500,
    "annotations": ["forte_pente", "lacets"],
    "metrics": {
        "mean_slope_pct": 17.2,
        "direction_changes": 4
    },
}

print(humanize_portion(portion))

## 3. Verbalisation d'un itinéraire entier

On peut ensuite verbaliser chaque portion séparément.
Une étape ultérieure pourra fusionner les phrases en un paragraphe global.

In [ ]:
def humanize_route(portions):
    return [
        {
            "start_m": p["start_m"],
            "end_m": p["end_m"],
            "text": humanize_portion(p),
        }
        for p in portions
    ]